# Wikimedia

In [8]:
import requests

url = "https://wikimedia.org/api/rest_v1/metrics/pageviews/top-per-country/CA/all-access/2026/08/01"
resp = requests.get(url, headers={"User-Agent": "zitygeist-exploration (personal project)"})
print(resp.status_code)
data = resp.json()

200


## Raw response shape

In [2]:
data.keys()

dict_keys(['items'])

In [3]:
data['items'][0]['articles'][0]

{'article': 'Main_Page',
 'project': 'en.wikipedia',
 'views_ceil': 169000,
 'rank': 1}

## Noise filtering

Some ranked "articles" aren't content — search pages, main pages, portals. Need a rule to tell those apart from real articles.

### Naive colon filter — false positives

First instinct: filter anything with a colon (`Special:Search`, `Wikipedia:...`). Doesn't work — real titles have colons too (`Spider-Man:_Brand_New_Day`).

In [4]:
for item in data['items'][0]['articles']:
    if ":" in item['article']:
        print(item['article'], item['rank'])

Spider-Man:_Brand_New_Day 2
Special:Search 3
Wikipedia:Featured_pictures 4
Wikipédia:Accueil_principal 7
UFC_Fight_Night:_Medić_vs._Rodriguez 22
Spider-Man:_No_Way_Home 28
Wikipedia:首页 29
Portal:Current_events 39
Avengers:_Doomsday 54
Spécial:Recherche 67
Spider-Man:_Homecoming 68
Wiktionary:Main_Page 70
Spider-Man:_Far_From_Home 92
Special:Search 97
Marvel_Cinematic_Universe:_Phase_Six 114
Avatar_Aang:_The_Last_Airbender 120
Special:UrlShortener 125
Batman:_Caped_Crusader 126
Avengers:_Secret_Wars 208


### Namespace-driven filter

Better rule: ask each project's own API what its real namespace names and main page title are, instead of guessing. A colon prefix only counts as noise if it matches a *real* namespace name for that specific project (namespace names are localized per language, e.g. `Wikipédia:` vs `Wikipedia:`).

Logic lives in `wikimedia_noise.py` (next to this notebook) so it's reusable, not copy-pasted per notebook.

In [5]:
projects = sorted({a['project'] for a in data['items'][0]['articles']})
projects

['commons.wikimedia',
 'en.wikipedia',
 'en.wiktionary',
 'fr.wikipedia',
 'wikitech.wikimedia',
 'zh.wikipedia',
 'zu.wikibooks']

In [9]:
from wikimedia_noise import fetch_siteinfo, is_noise

for p in projects:
    fetch_siteinfo(p)  # warm cache, so we can inspect below

{p: fetch_siteinfo(p)['mainpage'] for p in projects}

{'commons.wikimedia': 'Main_Page',
 'en.wikipedia': 'Main_Page',
 'en.wiktionary': 'Wiktionary:Main_Page',
 'fr.wikipedia': 'Wikipédia:Accueil_principal',
 'wikitech.wikimedia': 'Main_Page',
 'zh.wikipedia': 'Wikipedia:首页',
 'zu.wikibooks': 'Ikhasi_Elikhulu'}

In [8]:
articles = data['items'][0]['articles']
noise = [a for a in articles if is_noise(a['article'], a['project'])]
kept = [a for a in articles if not is_noise(a['article'], a['project'])]

print(f"{len(articles)} total -> {len(noise)} noise, {len(kept)} kept")
for a in noise:
    print(' ', a['rank'], a['project'], a['article'])

220 total -> 12 noise, 208 kept
  1 en.wikipedia Main_Page
  3 en.wikipedia Special:Search
  4 en.wikipedia Wikipedia:Featured_pictures
  7 fr.wikipedia Wikipédia:Accueil_principal
  15 wikitech.wikimedia Main_Page
  29 zh.wikipedia Wikipedia:首页
  39 en.wikipedia Portal:Current_events
  44 commons.wikimedia Main_Page
  67 fr.wikipedia Spécial:Recherche
  70 en.wiktionary Wiktionary:Main_Page
  97 zu.wikibooks Special:Search
  125 zu.wikibooks Special:UrlShortener


### Remaining edge cases — not caught by the namespace rule

`Deaths_in_2026` (rank 13) is a real article, so the namespace rule correctly keeps it — but it's the same kind of structurally-recurring, low-signal page the insights doc flagged. This isn't a namespace problem, it's a title-pattern problem, so it needs its own small rule.

In [9]:
import re

RECURRING_PATTERNS = [
    re.compile(r"^Deaths_in_\d{4}$"),
]

def is_recurring(article: str) -> bool:
    return any(p.match(article) for p in RECURRING_PATTERNS)

[a['article'] for a in kept if is_recurring(a['article'])]

['Deaths_in_2026']